# F5-TTS-THAI — สร้างเสียงบรรยายคลิป `วิธีทำคลิปด้วย Claude Design`

เปิดบน **Google Colab** และเลือก Runtime > Change runtime type > **T4 GPU** แล้วกด Run All

ผลลัพธ์: `voice.wav` (60 วินาที 6 ประโยค) — ดาวน์โหลดแล้วใส่ใน repo → รัน `python merge_audio.py` เพื่อ mux กับ `clip_silent.mp4`

In [ ]:
# 1) ติดตั้ง F5-TTS-THAI
!pip install -q f5-tts-th soundfile numpy

In [ ]:
# 2) เตรียม reference audio ภาษาไทย
# ใช้ตัวอย่างเสียงจาก repo F5-TTS-THAI เอง
import urllib.request, os
REF_URL = "https://github.com/VYNCX/F5-TTS-THAI/raw/main/examples/thai_example.wav"
REF_TEXT = "ได้รับข่าวคราวของเราที่จะหาที่มันเป็นไปที่จะจัดขึ้น."
urllib.request.urlretrieve(REF_URL, "ref.wav")
# ถ้าอยากเปลี่ยน voice ให้อัดเสียงตัวเอง 5-10 วินาที แล้วแทน ref.wav + ref_text
print("ok")

In [ ]:
# 3) สคริปต์ภาษาไทย 6 ประโยค (แนว hype)
LINES = [
    "เฮ้ อยากทำคลิปโซเชียลสวยๆ ใน หนึ่งนาทีมั้ย วันนี้คล็อดดีไซน์ มาช่วยแล้ว",
    "ขั้นตอนแรก แค่บอกคล็อด ว่าอยากได้คลิปแบบไหน พิมพ์ธีมแล้วรอเลย",
    "สอง เลือกแนวตั้ง เก้าต่อสิบหก เหมาะกับ ติ๊กต๊อก รีล และชอร์ต เป๊ะมาก",
    "สาม ใส่ข้อความบรรยาย ฟอนต์ สี และภาพประกอบ สวยปังในไม่กี่คลิก",
    "สี่ กดส่งออกเป็นไฟล์วิดีโอ แล้วโพสต์ได้เลย ทุกแพลตฟอร์ม",
    "ถ้าชอบ อย่าลืมกดไลก์ กดติดตาม แล้วเจอกันคลิปหน้าเลยนะ"
]

In [ ]:
# 4) สังเคราะห์แต่ละประโยค
from f5_tts_th.tts import TTS
import soundfile as sf
import numpy as np, subprocess

tts = TTS(model="v2")  # v2 = IPA, ลดอ่านข้าม/ซ้ำคำ

for i, text in enumerate(LINES, 1):
    wav = tts.infer(
        ref_audio="ref.wav",
        ref_text=REF_TEXT,
        gen_text=text,
        step=32, cfg=2.0, speed=1.15  # 1.15 เร็วขึ้นให้ดูตื่นเต้น
    )
    sf.write(f"line{i}.wav", wav, 24000)
    print(f"line{i}: {len(wav)/24000:.2f}s")

In [ ]:
# 5) รวมเป็น voice.wav ความยาว 60s วางตรงฉาก (ทุก 10 วินาที)
!apt-get install -y -qq ffmpeg > /dev/null
import subprocess

inputs = sum([["-i", f"line{i}.wav"] for i in range(1,7)], [])
fc_parts = []
for i in range(6):
    delay = i*10_000 + 300
    fc_parts.append(f"[{i}:a]adelay={delay}|{delay},apad[a{i}]")
fc_parts.append("".join(f"[a{i}]" for i in range(6)) + "amix=inputs=6:normalize=0,volume=1.3,atrim=0:60,loudnorm=I=-14:TP=-1[aout]")
fc = ";".join(fc_parts)

subprocess.run(["ffmpeg","-y",*inputs,"-filter_complex",fc,"-map","[aout]","-c:a","pcm_s16le","-ar","44100","voice.wav"], check=True)
print("voice.wav ready — dur:", subprocess.check_output(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0","voice.wav"]).decode().strip())

In [ ]:
# 6) ดาวน์โหลด voice.wav
from google.colab import files
files.download("voice.wav")